This is an educational implementation of z-codes (aka z-values aka morton codes).

z-codes are an example of space-filling curve (sfc), i.e. they provide a linearization order for multidimensional data.

In this notebook on mapping two-dimensional data to a one dimensional space, we use a simple z-code implementation to illustrate the concept of space-filling curves. However, z-codes can also easily be adapted to higher dimensions.

Other examples of space-filling curves are the Hilbert curve, Peano curve, or the Sierpinski curve.


In [1]:
from operator import add


def scalar_to_z_code(recursion_level: int, x: int, y: int) -> str:
    """
    Converts scalar coordinates into a z-code.

    Notice that we use a string-representation of z-codes for educational purposes
    a real implementation would represent z-codes in a bit-vector or map them to signed integers
    :param recursion_level: the recursion_level of the z-code
    :param x: x-coordinate
    :param y: y-coordinate
    :return: z-code as a string
    """
    if recursion_level == 0:
        # return empty string for recursion_level 0
        return ""

    # get the binary representation of x and y as a string and prefix them with zeros:
    x_binary = bin(x)[2:].zfill(recursion_level)
    y_binary = bin(y)[2:].zfill(recursion_level)

    assert len(x_binary) == len(y_binary) == recursion_level

    # zip the binary representations of x and y and interleave them#
    # this is the actual "trick" of z-codes: the bits of the n=2 binary numbers are interleaved
    ret = "".join(map(add, y_binary, x_binary))

    assert len(ret) == 2 * recursion_level

    return ret


scalar_to_z_code(3, 2, 1)

'000110'

In [2]:
def z_code_to_scalar(z_code: str) -> tuple[int, int, int]:
    """
    inverse of scalar_to_z_code function

    converts z-code to scalar-coordinates:
    :param z_code: string-representation of a z-code
    :return: recursion_level, scalar x, scalar y
    """
    assert len(z_code) % 2 == 0

    dimensions: int = 2

    recursion_level: int = len(z_code) // dimensions
    x: int = 0
    y: int = 0
    exponent: int = int(recursion_level) - 1

    i: int
    for i in range(0, len(z_code), 2):
        # reconstruct x and y by reversing the interleaving of the z-code
        # basically builds sum of 2^i exponents
        y += int(z_code[i]) * 2**exponent
        x += int(z_code[i + 1]) * 2**exponent
        exponent -= 1

    return recursion_level, x, y


z_code_to_scalar("000110")

(3, 2, 1)

In [3]:
z_code_to_scalar(scalar_to_z_code(2, 3, 1))

(2, 3, 1)

In [4]:
startEsc: str = "\033[1;"
middleEsc: str = "m"
endEsc: str = "\033[1;m"


def color_code(string: str, stepSize: int = 2):
    """color-codes a string in steps of <stepSize> with different colors

    :param string: the string to color-code
    :param stepSize: the step-size for coloring
    """
    color: int = 31
    i: int
    for i in range(0, len(string), stepSize):
        prefix: str = string[i : i + stepSize]
        print(startEsc + str(color) + middleEsc + prefix + endEsc, end="")
        color += 1


color_code("hello world")

hello world

In [5]:
def print_z_codes(recursion_level: int):
    """print all z-codes of a given recursion_level

    :param recursion_level: the recursion level
    """
    y: int
    for y in range(2**recursion_level):
        x: int
        for x in range(2**recursion_level):
            z_code: str = scalar_to_z_code(recursion_level, x, y)
            color_code(z_code)
            print(" ", end="")
        print()


print_z_codes(3)

000000 000001 000100 000101 010000 010001 010100 010101 
000010 000011 000110 000111 010010 010011 010110 010111 
001000 001001 001100 001101 011000 011001 011100 011101 
001010 001011 001110 001111 011010 011011 011110 011111 
100000 100001 100100 100101 110000 110001 110100 110101 
100010 100011 100110 100111 110010 110011 110110 110111 
101000 101001 101100 101101 111000 111001 111100 111101 
101010 101011 101110 101111 111010 111011 111110 111111 


In [6]:
def compute_z_codes(recursion_level: int):
    """computes all z-codes for the given recursion_level

    @param recursion_level: the recursion level
    @return: a list of tuples containing the z-code, scalar x, and scalar y coordinates
    """
    results: list[tuple[str, int, int]] = []
    y: int
    for y in range(2**recursion_level):
        x: int
        for x in range(2**recursion_level):
            z_code: str = scalar_to_z_code(recursion_level, x, y)
            results.append((z_code, x, y))

    return results

In [7]:
from ipycanvas import Canvas, hold_canvas


def visualize_z_codes(recursion_level: int):
    """visualize the data space in z-code order, i.e. draw the z-curve

    :param recursion_level: the recursion level
    """
    results: list[tuple[str, int, int]] = compute_z_codes(recursion_level)
    results.sort()
    x_width: int = 300
    y_width: int = x_width
    canvas = Canvas(size=(x_width, y_width), sync_image_data=True)
    canvas.line_width = 1
    max_x: int = 2**recursion_level
    max_y: int = max_x
    mult_x: float = x_width / max_x
    mult_y: float = y_width / max_y
    start: tuple[str, int, int] = results.pop(0)
    start_x: int = start[1]
    start_y: int = start[2]

    with hold_canvas(canvas):
        while len(results) > 0:
            next: tuple[str, int, int] = results.pop(0)
            next_x: int = next[1]
            next_y: int = next[2]
            canvas.begin_path()
            canvas.move_to((start_x + 0.5) * mult_x, (start_y + 0.5) * mult_y)
            canvas.line_to((next_x + 0.5) * mult_x, (next_y + 0.5) * mult_y)
            canvas.stroke()
            start_x, start_y = next_x, next_y

    return canvas


# visualize the z-codes for recursion levels 1 to 5
for _recursion_level in range(1, 6):
    display(visualize_z_codes(_recursion_level))

Canvas(sync_image_data=True)

Canvas(sync_image_data=True)

Canvas(sync_image_data=True)

Canvas(sync_image_data=True)

Canvas(sync_image_data=True)